# MSS-Agent 多Agent协作教程

## 学习目标

学完后你能:
- 用 Quorum-Fast 检测多Agent群体收敛 (收敛=坏)
- 用 Elevation Protocol 解决Agent间冲突 (不是投票)
- 理解 MSS 多Agent = K3 多Agent 的根本差异

## 前置知识

已完成 [01_quickstart.ipynb](01_quickstart.ipynb)

In [ ]:
from mss_agent.protocols import QuorumFast, ElevationProtocol
from mss_agent import MSSAgent

## 1. Quorum-Fast: 收敛检测

K3 多Agent: 投票选最优 → 收敛是好 (多数一致)
MSS 多Agent: 收敛 = 群体思维 → 收敛是坏 (都一个看法)

**Quorum > 0.8 或 < 0.2 → HYPER_CONVERGENT → 告警**

In [ ]:
def demo_quorum():
    # 场景A: 健康 — 观点分歧
    qf = QuorumFast()
    qf.report("analyst_a", 0.72, "方案1: 微服务")
    qf.report("analyst_b", 0.35, "方案2: 单体")
    qf.report("analyst_c", 0.68, "方案3: 混合")
    print(f"健康场景: quorum={qf.quorum()} → {qf.status()}")
    print(f"  解读: 三个Agent各有看法, 健康的分歧. 不应投票归一.")
    print()
    
    # 场景B: 危险 — 群体思维
    qf2 = QuorumFast()
    qf2.report("analyst_a", 0.95, "方案1")
    qf2.report("analyst_b", 0.92, "方案1")
    qf2.report("analyst_c", 0.88, "方案1")
    print(f"危险场景: quorum={qf2.quorum()} → {qf2.status()}")
    print(f"  解读: 全部同意=群体思维. 可能都错了但没人质疑!")
    print(f"  快照: {qf2.snapshot()}")

demo_quorum()

## 2. Elevation Protocol: 升维解决冲突

普通解决: 投票 → 多数赢 (降维)
MSS 解决: 找到被困维度 → 加一维 → 冲突消失

经典例子:
- **被困维度**: 速度 vs 成本 (二元对立)
- **升维**: 加"旅行目的"维度
- **解决**: 商务旅行→速度优先; 度假→成本优先; 不再冲突

In [ ]:
ep = ElevationProtocol()

# 示例1: 代码风格
r = ep.resolve(
    "Team A: 用Tabs缩进 (效率高)",
    "Team B: 用Spaces缩进 (跨平台一致)",
    "Tabs vs Spaces, 选哪个?"
)
print("=== 代码风格 ===")
print(f"被困: {r['trapped_dim']}")
print(f"升维: {r['elevation']}")
print(f"方案: {r['resolution'][:120]}")
print()

# 示例2: 微服务 vs 单体
r2 = ep.resolve(
    "Architect A: 微服务 (独立部署, 弹性伸缩)",
    "Architect B: 单体 (开发快, 运维简单)",
    "微服务还是单体?"
)
print("=== 架构选型 ===")
print(f"被困: {r2['trapped_dim']}")
print(f"升维: {r2['elevation']}")
print(f"方案: {r2['resolution'][:120]}")
print()

# 示例3: 真正的冲突 — Agent之间
r3 = ep.resolve(
    "TravelAgent: 选直飞航班 (客户要快)",
    "CostAgent: 选转机航班 (预算管控)",
    "客户需求和预算冲突"
)
print("=== 业务冲突 ===")
print(f"被困: {r3['trapped_dim']}")
print(f"升维: {r3['elevation']}")
print(f"方案: {r3['resolution'][:120]}")

## 3. 升维 vs 投票: 根本差异

| | K3 投票 | MSS 升维 |
|---|---------|----------|
| 前提 | 已有最优解在选项里 | 最优解可能不在选项里 |
| 方法 | 数人头 | 找被困维度 |
| 结果 | 少数服从多数 | 冲突本身消失 |
| 风险 | 群体思维的胜利 | — |
| 适用 | 简单选择 | 复杂冲突 |

## 4. 生产环境: MSSOrchestrator

完整的多Agent编排器参见 [maf_integration_demo.py](../examples/maf_integration_demo.py):

```python
orch = MSSOrchestrator()
orch.register("travel", travel_agent)
orch.register("reviewer", review_agent)

# 广播 + quorum检测
results = orch.broadcast("设计REST API")
print(orch.quorum.status())  # → DIVERGENT (healthy)

# 冲突升维
resolution = orch.resolve_conflict("travel", "reviewer", "速度vs成本")
```

## 下一步

- [API Reference](https://mysama1.github.io/MSS-AI-Project/)
- [GitHub Discussions](https://github.com/mysama1/MSS-AI-Project/discussions)